In [1]:
#!/usr/bin/env python3
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from arch import arch_model


# ============================================================
# Core stats
# ============================================================
def sharpe_hat(x: np.ndarray) -> float:
    x = np.asarray(x, float)
    s = x.std(ddof=1)
    return np.nan if (not np.isfinite(s) or s <= 0) else x.mean() / s


def wald_ci(S_hat: float, se: float, alpha: float) -> tuple[float, float]:
    z = NormalDist().inv_cdf(1.0 - alpha / 2.0)
    return (S_hat - z * se, S_hat + z * se)


# ============================================================
# DGPs
# ============================================================
def simulate_iid_normal(rng: np.random.Generator, n: int, S_true: float, sigma: float = 1.0) -> np.ndarray:
    mu = S_true * sigma
    return mu + sigma * rng.standard_normal(n)


def simulate_garch11_t(
    rng: np.random.Generator,
    n: int,
    S_true: float,
    g_alpha: float,
    g_beta: float,
    nu: float,
    sigma_uncond: float = 1.0,
    burn: int = 500,
) -> np.ndarray:
    if g_alpha + g_beta >= 1:
        raise ValueError("Need g_alpha + g_beta < 1 for finite unconditional variance.")

    omega = (1.0 - g_alpha - g_beta) * (sigma_uncond**2)
    mu = S_true * sigma_uncond

    m = burn + n
    h = np.empty(m, float)
    eps = np.empty(m, float)
    x = np.empty(m, float)

    # Standardize t innovations to Var(z)=1
    z = rng.standard_t(df=nu, size=m) * np.sqrt((nu - 2.0) / nu)

    h[0] = sigma_uncond**2
    eps[0] = np.sqrt(h[0]) * z[0]
    x[0] = mu + eps[0]

    for t in range(1, m):
        h[t] = omega + g_alpha * (eps[t - 1] ** 2) + g_beta * h[t - 1]
        eps[t] = np.sqrt(h[t]) * z[t]
        x[t] = mu + eps[t]

    return x[burn:]


# ============================================================
# SE estimators
# ============================================================
def se_iid_analytic(S_hat: float, n: int) -> float:
    # iid normal plug-in: Var(S_hat) ~ (1 + 0.5 S^2) / n
    return float(np.sqrt((1.0 + 0.5 * S_hat**2) / n))


def se_bootstrap_iid(x: np.ndarray, rng: np.random.Generator, n_boot: int) -> float:
    x = np.asarray(x, float)
    n = len(x)
    idx = rng.integers(0, n, size=(n_boot, n))
    xb = x[idx]
    m = xb.mean(axis=1)
    s = xb.std(axis=1, ddof=1)
    Sb = m / s
    return float(np.nanstd(Sb, ddof=1))


def _simulate_from_fit(res, rng: np.random.Generator, nobs: int, burn: int) -> np.ndarray:
    """
    arch's simulate signature differs across versions.
    Try rng=Generator, then rng=RandomState, then fallback to global seed.
    """
    try:
        sim = res.model.simulate(res.params, nobs=nobs, burn=burn, rng=rng)
        return np.asarray(sim["data"], float)
    except TypeError:
        pass

    try:
        rs = np.random.RandomState(int(rng.integers(0, 2**32 - 1)))
        sim = res.model.simulate(res.params, nobs=nobs, burn=burn, rng=rs)
        return np.asarray(sim["data"], float)
    except TypeError:
        pass

    # last resort: seed global RNG and call without rng
    np.random.seed(int(rng.integers(0, 2**32 - 1)))
    sim = res.model.simulate(res.params, nobs=nobs, burn=burn)
    return np.asarray(sim["data"], float)


def se_bootstrap_garch_parametric(
    x: np.ndarray,
    rng: np.random.Generator,
    n_boot: int,
    burn: int,
) -> float:
    x = np.asarray(x, float)

    am = arch_model(x, mean="Constant", vol="GARCH", p=1, q=1, dist="StudentsT")
    res = am.fit(disp="off")

    S_star = np.empty(n_boot, float)
    for b in range(n_boot):
        x_star = _simulate_from_fit(res, rng=rng, nobs=len(x), burn=burn)
        S_star[b] = sharpe_hat(x_star)

    return float(np.nanstd(S_star, ddof=1))


# ============================================================
# Experiment config + deterministic seeding
# ============================================================
@dataclass(frozen=True)
class Config:
    seed: int = 0
    alpha: float = 0.05

    R: int = 300
    n_boot: int = 300

    dgps: tuple[str, ...] = ("iid_normal", "garch11_t")
    methods: tuple[str, ...] = ("iid_normal_analytic", "bootstrap")

    n_grid: tuple[int, ...] = (36, 60, 120)
    S_grid: tuple[float, ...] = (0.2, 0.4, 0.6, 0.8, 1.0)

    # GARCH truth
    g_alpha: float = 0.05
    g_beta: float = 0.90
    nu: float = 7.0
    burn: int = 500

    # parallel
    max_workers: int = max(1, (os.cpu_count() or 2) - 1)


def stable_seed(*parts: object) -> int:
    s = "|".join(str(p) for p in parts)
    h = hashlib.sha256(s.encode("utf-8")).hexdigest()
    return int(h[:16], 16)  # 64-bit-ish


def coverage_ci_lower_approx(R: int, nominal: float, alpha: float = 0.05) -> float:
    # normal approx lower bound for Binomial(R, nominal)/R
    z = NormalDist().inv_cdf(1.0 - alpha / 2.0)
    se = np.sqrt(nominal * (1.0 - nominal) / R)
    return float(nominal - z * se)


# ============================================================
# One (dgp, n, S_true) cell (runs inside a worker process)
# ============================================================
def run_cell(dgp: str, n: int, S_true: float, cfg_dict: dict) -> list[dict]:
    cfg = Config(**cfg_dict)
    rng = np.random.default_rng(stable_seed(cfg.seed, dgp, n, S_true))

    S_hats: list[float] = []

    se_A: list[float] = []
    cov_A: list[float] = []
    rej_A: list[float] = []

    se_B: list[float] = []
    cov_B: list[float] = []
    rej_B: list[float] = []

    for _ in range(cfg.R):
        if dgp == "iid_normal":
            x = simulate_iid_normal(rng, n=n, S_true=S_true, sigma=1.0)
        elif dgp == "garch11_t":
            x = simulate_garch11_t(
                rng,
                n=n,
                S_true=S_true,
                g_alpha=cfg.g_alpha,
                g_beta=cfg.g_beta,
                nu=cfg.nu,
                sigma_uncond=1.0,
                burn=cfg.burn,
            )
        else:
            raise ValueError(f"Unknown dgp: {dgp}")

        S_hat = sharpe_hat(x)
        S_hats.append(S_hat)

        # Method A: iid analytic
        se = se_iid_analytic(S_hat, n)
        lo, hi = wald_ci(S_hat, se, cfg.alpha)
        se_A.append(se)
        cov_A.append(float(lo <= S_true <= hi))
        rej_A.append(float(lo > 0.0))

        # Method B: bootstrap (iid nonparam for iid_normal; parametric GARCH for garch11_t)
        try:
            if dgp == "iid_normal":
                se = se_bootstrap_iid(x, rng, n_boot=cfg.n_boot)
            else:
                se = se_bootstrap_garch_parametric(x, rng, n_boot=cfg.n_boot, burn=cfg.burn)

            lo, hi = wald_ci(S_hat, se, cfg.alpha)
            se_B.append(se)
            cov_B.append(float(lo <= S_true <= hi))
            rej_B.append(float(lo > 0.0))
        except Exception:
            se_B.append(np.nan)
            cov_B.append(np.nan)
            rej_B.append(np.nan)

    S_arr = np.asarray(S_hats, float)
    mc_sd = float(np.nanstd(S_arr, ddof=1))
    bias = float(np.nanmean(S_arr - S_true))
    rmse = float(np.sqrt(np.nanmean((S_arr - S_true) ** 2)))

    def pack(method: str, se_list, cov_list, rej_list) -> dict:
        se_arr = np.asarray(se_list, float)
        cov_arr = np.asarray(cov_list, float)
        rej_arr = np.asarray(rej_list, float)
        return {
            "dgp": dgp,
            "n": int(n),
            "S_true": float(S_true),
            "method": method,
            "bias": bias,
            "rmse": rmse,
            "coverage_95": float(np.nanmean(cov_arr)),
            "reject_rate_H0_S_le_0": float(np.nanmean(rej_arr)),
            "se_ratio_meanSE_over_mcSD": float(np.nanmean(se_arr) / mc_sd) if mc_sd > 0 else np.nan,
            "mc_sd_S_hat": mc_sd,
            "fit_fail_rate": float(np.mean(np.isnan(se_arr))),
        }

    return [
        pack("iid_normal_analytic", se_A, cov_A, rej_A),
        pack("bootstrap", se_B, cov_B, rej_B),
    ]


# ============================================================
# Run + cache
# ============================================================
def run_partA(cfg: Config) -> pd.DataFrame:
    tasks = [(dgp, n, s) for dgp in cfg.dgps for n in cfg.n_grid for s in cfg.S_grid]
    cfg_dict = asdict(cfg)

    rows: list[dict] = []
    if cfg.max_workers <= 1:
        for (dgp, n, s) in tasks:
            rows.extend(run_cell(dgp, n, s, cfg_dict))
    else:
        with ProcessPoolExecutor(max_workers=cfg.max_workers) as ex:
            futs = [ex.submit(run_cell, dgp, n, s, cfg_dict) for (dgp, n, s) in tasks]
            for fut in as_completed(futs):
                rows.extend(fut.result())

    df = pd.DataFrame(rows)
    return df.sort_values(["dgp", "n", "S_true", "method"]).reset_index(drop=True)


# ============================================================
# Plots (same structure as your (2), no HAC anywhere)
# ============================================================
def save_fig(fig: plt.Figure, fig_dir: Path, stem: str) -> None:
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_dir / f"{stem}.png", dpi=160, bbox_inches="tight")
    fig.savefig(fig_dir / f"{stem}.pdf", bbox_inches="tight")


def plot_all(results: pd.DataFrame, cfg: Config, fig_dir: Path) -> None:
    coverage_metric = "coverage_95"
    coverage_nominal = 0.95
    coverage_ci_lower = coverage_ci_lower_approx(cfg.R, coverage_nominal, alpha=0.05)

    coverage_min = min(float(results[coverage_metric].min()), coverage_ci_lower)
    coverage_max = max(float(results[coverage_metric].max()), coverage_nominal)
    if np.isclose(coverage_min, coverage_max):
        coverage_min -= 0.01
        coverage_max += 0.01

    n_values = sorted(int(v) for v in results["n"].unique())
    s_values = sorted(float(v) for v in results["S_true"].unique())

    # --- Coverage by S_true (lines are n), panel is dgp x method
    fig, axes = plt.subplots(
        len(cfg.dgps),
        len(cfg.methods),
        figsize=(4.5 * len(cfg.methods), 3.3 * len(cfg.dgps)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    for i, dgp in enumerate(cfg.dgps):
        for j, method in enumerate(cfg.methods):
            ax = axes[i, j]
            subset = results[(results["dgp"] == dgp) & (results["method"] == method)]
            for n in n_values:
                n_rows = subset[subset["n"] == int(n)].sort_values("S_true")
                n_rows = n_rows.dropna(subset=[coverage_metric])
                if n_rows.empty:
                    continue
                ax.plot(n_rows["S_true"], n_rows[coverage_metric], marker="o", label=f"n={n}")
            ax.axhline(coverage_nominal, linestyle=":", color="black", linewidth=1.0)
            ax.axhline(
                coverage_ci_lower,
                linestyle="-",
                color="red",
                linewidth=1.0,
                label="95% CI lower bound" if (i == 0 and j == 0) else None,
            )
            ax.grid(alpha=0.25, linestyle=":")
            ax.set_title(f"{dgp} | {method}")
            ax.set_xlabel("True Sharpe")
            ax.set_ylabel("95% coverage")
            ax.set_ylim(coverage_min, coverage_max)

    handles, labels = [], []
    for ax in axes.ravel():
        h, l = ax.get_legend_handles_labels()
        if h:
            handles, labels = h, l
            break
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 6))
    fig.tight_layout(rect=(0, 0, 1, 0.93 if handles else 1.0))
    save_fig(fig, fig_dir, "partA_coverage_95_by_n_grid")
    plt.close(fig)

    # --- Coverage vs n at fixed S_true (one column, dgp rows)
    target_s_true = 0.5
    if s_values and not any(np.isclose(v, target_s_true) for v in s_values):
        target_s_true = min(s_values, key=lambda v: abs(v - target_s_true))

    fig2, axes2 = plt.subplots(
        len(cfg.dgps),
        1,
        figsize=(6.0, 3.3 * len(cfg.dgps)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    for i, dgp in enumerate(cfg.dgps):
        ax = axes2[i, 0]
        subset = results[(results["dgp"] == dgp) & (np.isclose(results["S_true"], float(target_s_true)))]
        for method in cfg.methods:
            m_rows = subset[subset["method"] == method].sort_values("n")
            m_rows = m_rows.dropna(subset=[coverage_metric])
            if m_rows.empty:
                continue
            ax.plot(m_rows["n"], m_rows[coverage_metric], marker="o", label=method)
        ax.axhline(coverage_nominal, linestyle=":", color="black", linewidth=1.0)
        ax.axhline(
            coverage_ci_lower,
            linestyle="-",
            color="red",
            linewidth=1.0,
            label="95% CI lower bound" if i == 0 else None,
        )
        ax.grid(alpha=0.25, linestyle=":")
        ax.set_title(f"{dgp} | S_true={target_s_true:g}")
        ax.set_xlabel("n")
        ax.set_ylabel("95% coverage")
        ax.set_ylim(coverage_min, coverage_max)

    handles2, labels2 = [], []
    for ax in axes2.ravel():
        h, l = ax.get_legend_handles_labels()
        if h:
            handles2, labels2 = h, l
            break
    if handles2:
        fig2.legend(handles2, labels2, loc="upper center", ncol=min(len(labels2), 6))
    fig2.tight_layout(rect=(0, 0, 1, 0.93 if handles2 else 1.0))
    save_fig(fig2, fig_dir, f"partA_coverage_95_vs_n_s_true_{str(target_s_true).replace('.', 'p')}")
    plt.close(fig2)

    # --- RMSE and Bias plots (same layouts as your (2))
    for metric_col, metric_label in [("rmse", "RMSE"), ("bias", "Bias")]:
        metric_min = float(results[metric_col].min())
        metric_max = float(results[metric_col].max())
        if metric_col == "bias":
            metric_min = min(metric_min, 0.0)
            metric_max = max(metric_max, 0.0)
        if np.isclose(metric_min, metric_max):
            pad = 0.01 if np.isclose(metric_max, 0.0) else abs(metric_max) * 0.05
            metric_min -= pad
            metric_max += pad

        # metric vs S_true (lines are n), panel is dgp x method
        fig_s, axes_s = plt.subplots(
            len(cfg.dgps),
            len(cfg.methods),
            figsize=(4.5 * len(cfg.methods), 3.3 * len(cfg.dgps)),
            sharex=True,
            sharey=True,
            squeeze=False,
        )
        for i, dgp in enumerate(cfg.dgps):
            for j, method in enumerate(cfg.methods):
                ax = axes_s[i, j]
                subset = results[(results["dgp"] == dgp) & (results["method"] == method)]
                for n in n_values:
                    n_rows = subset[subset["n"] == int(n)].sort_values("S_true")
                    n_rows = n_rows.dropna(subset=[metric_col])
                    if n_rows.empty:
                        continue
                    ax.plot(n_rows["S_true"], n_rows[metric_col], marker="o", label=f"n={n}")
                if metric_col == "bias":
                    ax.axhline(0.0, linestyle=":", color="black", linewidth=1.0)
                ax.grid(alpha=0.25, linestyle=":")
                ax.set_title(f"{dgp} | {method}")
                ax.set_xlabel("True Sharpe")
                ax.set_ylabel(metric_label)
                ax.set_ylim(metric_min, metric_max)

        handles_s, labels_s = [], []
        for ax in axes_s.ravel():
            h, l = ax.get_legend_handles_labels()
            if h:
                handles_s, labels_s = h, l
                break
        if handles_s:
            fig_s.legend(handles_s, labels_s, loc="upper center", ncol=min(len(labels_s), 6))
        fig_s.tight_layout(rect=(0, 0, 1, 0.93 if handles_s else 1.0))
        save_fig(fig_s, fig_dir, f"partA_{metric_col}_vs_s_true_by_n_grid")
        plt.close(fig_s)

        # metric vs n (lines are S_true), panel is dgp x method
        fig_n, axes_n = plt.subplots(
            len(cfg.dgps),
            len(cfg.methods),
            figsize=(4.5 * len(cfg.methods), 3.3 * len(cfg.dgps)),
            sharex=True,
            sharey=True,
            squeeze=False,
        )
        for i, dgp in enumerate(cfg.dgps):
            for j, method in enumerate(cfg.methods):
                ax = axes_n[i, j]
                subset = results[(results["dgp"] == dgp) & (results["method"] == method)]
                for s_true in s_values:
                    s_rows = subset[np.isclose(subset["S_true"], float(s_true))].sort_values("n")
                    s_rows = s_rows.dropna(subset=[metric_col])
                    if s_rows.empty:
                        continue
                    ax.plot(s_rows["n"], s_rows[metric_col], marker="o", label=f"S_true={s_true:g}")
                if metric_col == "bias":
                    ax.axhline(0.0, linestyle=":", color="black", linewidth=1.0)
                ax.grid(alpha=0.25, linestyle=":")
                ax.set_title(f"{dgp} | {method}")
                ax.set_xlabel("n")
                ax.set_ylabel(metric_label)
                ax.set_ylim(metric_min, metric_max)

        handles_n, labels_n = [], []
        for ax in axes_n.ravel():
            h, l = ax.get_legend_handles_labels()
            if h:
                handles_n, labels_n = h, l
                break
        if handles_n:
            fig_n.legend(handles_n, labels_n, loc="upper center", ncol=min(len(labels_n), 6))
        fig_n.tight_layout(rect=(0, 0, 1, 0.93 if handles_n else 1.0))
        save_fig(fig_n, fig_dir, f"partA_{metric_col}_vs_n_by_s_true_grid")
        plt.close(fig_n)


def try_write_parquet(df: pd.DataFrame, path: Path) -> None:
    try:
        df.to_parquet(path, index=False)
    except Exception:
        pass


def env_versions() -> dict:
    try:
        import importlib.metadata as ilmd
    except Exception:
        ilmd = None

    pkgs = ["numpy", "pandas", "matplotlib", "arch", "pyarrow"]
    versions = {}
    for p in pkgs:
        try:
            versions[p] = ilmd.version(p) if ilmd else None
        except Exception:
            versions[p] = None
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": versions,
    }


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    try:
        import multiprocessing as mp

        mp.set_start_method("spawn", force=False)
    except Exception:
        pass

    run_dir = Path(os.environ.get("EXPERIMENT_BSC_OUTPUT_DIR", "outputs/experiment_bsc")).resolve()
    cache_dir = run_dir / "cache"
    fig_dir = run_dir / "figures"
    for p in (run_dir, cache_dir, fig_dir):
        p.mkdir(parents=True, exist_ok=True)

    cfg = Config(
        seed=int(os.environ.get("SEED", "0")),
        R=int(os.environ.get("R", "300")),
        n_boot=int(os.environ.get("N_BOOT", "300")),
        max_workers=int(os.environ.get("MAX_WORKERS", str(Config().max_workers))),
    )

    cfg_payload = json.dumps(asdict(cfg), sort_keys=True, separators=(",", ":"))
    cfg_hash = hashlib.sha256(cfg_payload.encode("utf-8")).hexdigest()[:12]
    cache_csv = cache_dir / f"partA_results_{cfg_hash}.csv"
    cache_parquet = cache_dir / f"partA_results_{cfg_hash}.parquet"

    if cache_csv.exists():
        results = pd.read_csv(cache_csv)
        print(f"[cache] loaded {cache_csv}")
    else:
        results = run_partA(cfg)
        results.to_csv(cache_csv, index=False)
        try_write_parquet(results, cache_parquet)
        (run_dir / f"config_partA_{cfg_hash}.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")
        (run_dir / f"env_versions_{cfg_hash}.json").write_text(json.dumps(env_versions(), indent=2), encoding="utf-8")
        print(f"[run] wrote {cache_csv}")

    # also write a “latest” copy for convenience
    results.to_csv(run_dir / "results_partA_bootstrap.csv", index=False)

    plot_all(results, cfg, fig_dir)

    print(results)
    print(f"\nResults: {run_dir / 'results_partA_bootstrap.csv'}")
    print(f"Figures: {fig_dir}")

/workspaces/finance-data-download-test/.venv/lib/python3.12/site-packages/arch/univariate/mean.py:1216: InitialValueWarning: Parameters are not consistent with a stationary model. Using the intercept
to initialize the model.

  sim_values = self.volatility.simulate(
/workspaces/finance-data-download-test/.venv/lib/python3.12/site-packages/arch/univariate/mean.py:1216: InitialValueWarning: Parameters are not consistent with a stationary model. Using the intercept
to initialize the model.

  sim_values = self.volatility.simulate(
/workspaces/finance-data-download-test/.venv/lib/python3.12/site-packages/arch/univariate/mean.py:1216: InitialValueWarning: Parameters are not consistent with a stationary model. Using the intercept
to initialize the model.

  sim_values = self.volatility.simulate(
/workspaces/finance-data-download-test/.venv/lib/python3.12/site-packages/arch/univariate/mean.py:1216: InitialValueWarning: Parameters are not consistent with a stationary model. Using the intercept

KeyboardInterrupt: 